# 07 Index Persistence and Refresh Lifecycle (LlamaIndex, 2026)

## What This Lesson Is
Persist retrieval state to disk and safely refresh indexes with new content.

## Scientific Lens
- Concept: Stateful index lifecycle management
- Measure: Consistency before/after reload and refresh success rate
- Validity Limit: Notebook persistence checks do not cover multi-node consistency or migration complexity.


## How It Works
1. Persist deterministic index state.
2. Reload and validate query consistency.
3. Apply live refresh and verify updated recall.


In [ ]:
from pathlib import Path
p = Path('/tmp/llamaindex_persist')
print("persist path:", p)


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: executes a real provider/CLI flow with explicit graceful-skip behavior.


In [ ]:
# Deterministic Demo
import json
from pathlib import Path

path = Path('/tmp/llamaindex_lifecycle_demo.json')
state = {"docs": [{"id": "D1", "text": "initial context"}]}
path.write_text(json.dumps(state))

loaded = json.loads(path.read_text())
loaded["docs"].append({"id": "D2", "text": "newly ingested context"})
path.write_text(json.dumps(loaded))

final = json.loads(path.read_text())
print(final)
assert len(final["docs"]) == 2


In [ ]:
# Live Demo
from pathlib import Path

try:
    from llama_index.core import Document, VectorStoreIndex, StorageContext, load_index_from_storage
except Exception as exc:
    print(f"Skipping live persistence demo: dependency missing ({exc})")
else:
    persist_dir = Path('/tmp/llamaindex_live_persist')
    docs = [Document(text="Initial reliability note.")]
    idx = VectorStoreIndex.from_documents(docs)
    idx.storage_context.persist(persist_dir=str(persist_dir))

    loaded = load_index_from_storage(StorageContext.from_defaults(persist_dir=str(persist_dir)))
    ans = str(loaded.as_query_engine(similarity_top_k=1).query("What is indexed?"))
    print(ans)
    assert ans


## Applied Labs
1. Add version metadata to persisted state and enforce version checks on load.
2. Simulate partial write failure and add recovery safeguards.
3. Measure query consistency before/after refresh on 10 prompts.

## Validation Checklist
- Persisted artifact is readable and versioned.
- Reloaded index can answer expected query without rebuild.
- Refresh path does not corrupt prior persisted state.

## Further Reading
- [LlamaIndex Persistence](https://docs.llamaindex.ai/en/stable/module_guides/storing/save_load/)
- [Data Versioning Concepts](https://martinfowler.com/articles/patterns-of-distributed-systems/versioned-value.html)
- [CAP Theorem Overview](https://www.allthingsdistributed.com/2008/12/eventually_consistent.html)
